In [1]:
import pandas as pd
import robustness as rob
from IPython.display import display, HTML
import warnings

def show_scroll(df, height=400):
    display(HTML(f'<div style="height:{height}px; overflow:auto">{df.to_html()}</div>'))

pd.set_option("display.max_rows", 200)       # kaç satır göstersin
pd.set_option("display.max_colwidth", None)  # left/right kırpılmasın
warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
from signifier_analysis import SignifierAnalyzer
import robustness as rob

# fit once in this cell with the correct values of n_components 
# for reproducibilty of following analyses
configs = {"demokrasi": 10, "millet": 10, "halk": 15, "hak": 12}
common = dict(ranking="both", max_df_ratio=1, dim_ratio=0.85,
              hide_common=True, top_words=100, root_len=5, min_word_len=3, verbose=False)

analyzers = {}
for word, nc in configs.items():
    a = SignifierAnalyzer(word, version="maskwhite")
    a.fit(n_components=nc, **common)
    analyzers[word] = a

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
I|19:17:47.399|Root lexicon created in 328 ms.                                                                     | DictionarySerializer#getDictionaryItems
I|19:17:47.400|Dictionary generated in 378 ms                                                                      | RootLexicon#defaultBinaryLexicon
I|19:17:47.662|Initialized in 687 ms.                                                                              | TurkishMorphology#createWithDefaults


In [ ]:
sig = {w: (a.v, a.clu, a.m["cluster"].values) for w, a in analyzers.items()}
sil_summary, _ = rob.report_all(sig)
print(sil_summary.to_string(index=False))

demokrasi | venue=hepsi | n=10911 | faz: {1: 4105, 2: 3066, 3: 3740}
number of clusters: 130 | noise: 19%
c59 n=  49 f1=0.0 f2=0.0 f3=2.9 [f3 2.9]
   freq: belediyeler(109; 36), başkan(60; 15), sosyal(53; 53), demokrat(51; 50), sodem(36; 23), konseyi(20; 17), partili(20; 7), birliği(19; 13), eşgüdüm(15; 14), ·türkiye(14; 14), genel(13), halk(13), hizmet(12), cumhuriyet(10), üzerinden(8), güdüm(7), istanbul(7), projeleri(7), ekrem(6), ortaklaştırarak(6), yapılar(5), yerel(5), yönetimlerden(5), önemli(5), chp(4), geçen(4), kararı(4), millete(4), seçimde(4), sistemden(4), sorumlu(4), toplanan(4), yardımcımız(4), yeni(4), örnek(4), özel(4), ayrı(3), başarılı(3), başında(3), biyolojik(3), bölge(3), büyükerşen(3), büyükşehir(3), elimizden(3), geleni(3), gökan(3), görevler(3), imamoğlu(3), kadrolarının(3), katkıları(3), kentler(3), mansur(3), metro(3), noktasında(3), sahip(3), salonda(3), sağlanması(3), sendikası(3), yılmaz(3), zeybek(3), özlem(3), şekilde(3), adaylaşmama(2), ajansı(2), arası

In [4]:
long_df, corr_df = rob.freq_dispersion_check(analyzers, force_neq=1199, seed=42)
print(long_df.to_string(index=False))
print(corr_df.to_string(index=False))

signifier  phase  freq   disp
demokrasi      1  3395 37.213
demokrasi      2  2522 36.944
demokrasi      3  2888 40.074
   millet      1  1694 36.441
   millet      2  2481 35.555
   millet      3  4298 39.469
     halk      1  1199 38.281
     halk      2  1513 37.074
     halk      3  1263 39.194
      hak      1  2325 36.776
      hak      2  3311 35.849
      hak      3  2582 41.467
    scope  n_points  pearson_r                              note
demokrasi         3     -0.015       within, 3 nokta, sadece yon
   millet         3      0.869       within, 3 nokta, sadece yon
     halk         3     -0.803       within, 3 nokta, sadece yon
      hak         3     -0.412       within, 3 nokta, sadece yon
   POOLED        12      0.094 p=0.772, spearman=0.154 (p=0.633)


In [4]:
from robustness import venue_dispersion_report
results = venue_dispersion_report(analyzers, n_runs=20, seed=42)
print(results)


=== DEMOKRASI  (ortak n_eq=133) ===
  rally       f1=36.44 f2=34.89 f3=37.64  [dom=f3*] ozel-kk=+1.20
  press       f1=36.74 f2=36.60 f3=40.96  [dom=f3*] ozel-kk=+4.22
  tv          f1=36.70 f2=37.09 f3=41.57  [dom=f3*] ozel-kk=+4.48
  parliament  f1=37.81 f2=37.66 f3=39.89  [dom=f3*] ozel-kk=+2.08
  event       f1=36.96 f2=36.20 f3=40.75  [dom=f3*] ozel-kk=+3.79

=== MILLET  (ortak n_eq=99) ===
  rally       f1=37.42 f2=33.20 f3=38.67  [dom=f3*] ozel-kk=+1.25
  press       f1=39.14 f2=35.77 f3=39.72  [dom=f3*] ozel-kk=+0.57
  tv          f1=36.06 f2=35.18 f3=38.79  [dom=f3*] ozel-kk=+2.73
  parliament  f1=35.68 f2=35.23 f3=40.20  [dom=f3*] ozel-kk=+4.52
  event       f1=35.76 f2=36.35 f3=38.62  [dom=f3*] ozel-kk=+2.28

=== HALK  (ortak n_eq=52) ===
  rally       f1=36.53 f2=36.80 f3=36.14  [dom=f2 ] ozel-kk=-0.66
  press       f1=37.52 f2=37.24 f3=40.33  [dom=f3*] ozel-kk=+2.82
  tv          f1=38.26 f2=37.70 f3=40.53  [dom=f3*] ozel-kk=+2.27
  parliament  f1=38.98 f2=37.40 f3=39.70 

In [4]:
from robustness import cross_signifier_dispersion
df, summary = cross_signifier_dispersion(analyzers)

global n_eq = 52  (butun signifier'lar bununla olculuyor)

=== signifier basi ortalama Ozel-KK genislemesi (dusukten yuksege) ===
            mean    min    max
signifier                     
halk       1.649 -0.606  2.922
millet     2.402  0.765  4.541
demokrasi  3.214  1.400  4.435
hak        5.108  3.816  6.323


In [3]:
from robustness import cross_signifier_dispersion_phase
df_phase = cross_signifier_dispersion_phase(analyzers)

global n_eq = 1199  (venue'suz, butun signifier'lar bununla)

=== Ozel-KK genislemesi (dusukten yuksege, ortak n_eq) ===
signifier  disp_f1  disp_f2  disp_f3  ozel_minus_kk
     halk   38.281   37.074   39.194          0.912
demokrasi   37.213   36.944   40.074          2.862
   millet   36.441   35.555   39.469          3.028
      hak   36.776   35.849   41.467          4.691


In [8]:
from robustness import freq_dispersion_check
print(freq_dispersion_check(analyzers))

(    signifier  phase  freq    disp
0   demokrasi      1  3395  37.125
1   demokrasi      2  2522  36.935
2   demokrasi      3  2888  40.049
3      millet      1  1694  36.374
4      millet      2  2481  35.581
5      millet      3  4298  39.358
6        halk      1  1199  38.340
7        halk      2  1513  36.993
8        halk      3  1263  39.178
9         hak      1  2325  36.760
10        hak      2  3311  35.835
11        hak      3  2582  41.596,        scope  n_points  pearson_r                               note
0  demokrasi         3     -0.039        within, 3 nokta, sadece yon
1     millet         3      0.878        within, 3 nokta, sadece yon
2       halk         3     -0.834        within, 3 nokta, sadece yon
3        hak         3     -0.407        within, 3 nokta, sadece yon
4     POOLED        12      0.081  p=0.802, spearman=0.154 (p=0.633))


In [4]:
hak_normal = SignifierAnalyzer("hak", version="normal")
hak_normal.fit(n_components=12, ranking="both", max_df_ratio=1, dim_ratio=0.85,
               hide_common=True, top_words=100, root_len=5, min_word_len=3)

print("normal UMAP shape:", hak_normal.clu.shape)   # (n, 12) olmalı
print("normal küme:", hak_normal.m["cluster"].nunique())
print("maskwhite küme:", analyzers["hak"].m["cluster"].nunique())
print()
print(rob.validate_mask_whiten(hak_normal, analyzers["hak"], form="hakkını").to_string(index=False))

hak | venue=hepsi | n=10009 | faz: {1: 2863, 2: 3983, 3: 3163}
number of clusters: 108 | noise: 13%
c28 n=  61 f1=0.1 f2=0.1 f3=2.9 [f3 2.9]
   freq: ·hakkına(73; 68), sahip(36; 34), kadının(20; 7), cumhuriyet(17; 15), darbe(15; 3), milletin(15; 7), partisi(15; 5), sandığa(14; 10), demokrasiye(13; 9), emaneti(10; 5), halk(10), meydana(10), türkiye(10), başkan(9), esenyurt(9), saygı(9), atatürk(8), hukukun(7), büyük(6), iktidar(6), insan(6), irade(6), kumpas(6), kürt(6), mart(6), saldırı(6), siyasi(6), özgürlüğüne(6), belediye(5), gazi(5), haber(5), hedefleri(5), istanbul(5), kemal(5), mitingimize(5), mustafa(5), beşiktaş(4), chp(4), demokratlara(4), eleştiri(4), izmir(4), kazanımı(4), medeni(4), mesele(4), müdahale(4), nafaka(4), savcılığını(4), adaletsizliğe(3), cinayet(3), cumhurbaşkanı(3), değerli(3), ekrem(3), tayyip(3), fetö(3), gelen(3), göz(3), kanun(3), kayyum(3), lazım(3), miras(3), muhalefet(3), mücadele(3), politikası(3), seçim(3), seçmenin(3), sokak(3), sonuna(3), tercihler

In [10]:
dem_normal = SignifierAnalyzer("demokrasi", version="normal")
dem_normal.fit(n_components=10, ranking="both", max_df_ratio=1, dim_ratio=0.85,
               hide_common=True, top_words=100, root_len=5, min_word_len=3)

print("normal küme:", dem_normal.m["cluster"].nunique())
print("maskwhite küme:", analyzers["demokrasi"].m["cluster"].nunique())

demokrasi | venue=hepsi | n=10911 | faz: {1: 4105, 2: 3066, 3: 3740}
number of clusters: 119 | noise: 24%
c85 n=  38 f1=0.1 f2=3.3 f3=0.2 [f2 3.3]
   freq: yollarla(39; 37), demokratik(38; 38), yönetimi(38; 30), tarihine(35; 17), gençler(32; 16), otoriter(32; 32), dünya(26; 24), siyaset(25; 18), sandığa(23; 21), ·türkiye(19; 19), armağan(13), güzel(11), ülkenin(10), miras(9), önemli(9), beraber(7), cumhuriyet(6), görevdir(4), hazır(4), huzur(4), kemal(4), oyunuzu(4), partisi(4), başkan(3), gelecek(3), ifade(3), kadınları(3), millet(3), mustafa(3), onuru(3), rektör(3), adam(2), alaşağı(2), açısından(2), baronları(2), baskıcı(2), başarı(2), başına(2), bini(2), davet(2), değişime(2), dikta(2), emekli(2), eylem(2), genel(2), gerçekleşti(2), gurur(2), halk(2), heyecanla(2), iktidarımızda(2), istanbul(2), kaderini(2), kimsenin(2), parlamentodan(2), rejimini(2), sahip(2), söz(2), temel(2), yağmur(2), yolu(2), örgütün(2), özgürlüklerden(2), adamdan(1), adayı(1), aklından(1), akşam(1), allah(1)

In [11]:
print(rob.validate_mask_whiten(dem_normal, analyzers["demokrasi"], form="demokrasinin").to_string(index=False))

'demokrasinin' toplam 999 kez, 38 kümede geçiyor
'demokrasinin' toplam 999 kez, 93 kümede geçiyor
            metric   normal  maskwhite
      total_tokens 743.0000    796.000
clusters_with_form  38.0000     93.000
               HHI   0.1231      0.053
             N_eff   8.1000     18.900
    clusters_90pct  11.0000     46.000
